In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point


In [3]:
station_shp = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\5. 1861 England, Wales and Scotland rail stations\1861EngWalesScotRail_Stations.shp"

accident_csv = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\detailed_accidents_data.csv"

branch_csv = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\ASRS\BalanceSheets\1875\Results\georeferenced_railway_results.csv"

# 1851 Registration Districts (England & Wales) - for uniform census-matchable geography
# Download from UK Data Service: https://reshare.ukdataservice.ac.uk/852948/
# Or Campop: https://www.campop.geog.cam.ac.uk/ (filename: 1851EngWalesRegistrationDistrict.shp)
# Place the .shp (and .shx, .dbf, .prj) in a folder and set path below:
rd_1851_shp = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\1851_RegistrationDistricts\1851EngWalesRegistrationDistrict.shp"

# Geocoding caches (if you have them from prior runs - used by R/Accidents_and_Unions)
base_path = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data"
accident_geo_cache = base_path + r"\cache_accident_geocoding.csv"
union_geo_cache = base_path + r"\cache_union_geocoding.csv"


In [4]:
stations = gpd.read_file(station_shp)

# Ensure projected CRS (meters); British National Grid
stations = stations.to_crs(epsg=27700)


In [9]:
print(stations.columns)


Index(['Id', 'geometry'], dtype='object')


In [10]:
pip install geopy tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Keitaro Ninomiya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd

acc = pd.read_csv(accident_csv)

acc["location_clean"] = (
    acc["location"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z ]", "", regex=True)
    .str.strip()
)


In [13]:
UK_KEYWORDS = [
    "england", "scotland", "wales", "london", "york",
    "junction", "station", "near", "at", "between"
]

def looks_uk(x):
    x = x.lower()
    return any(k in x for k in UK_KEYWORDS)

acc["location_clean"] = (
    acc["location"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z ]", "", regex=True)
    .str.strip()
)

acc["likely_uk"] = acc["location_clean"].apply(looks_uk)

locations = acc.loc[acc["likely_uk"], "location_clean"].unique()

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import time

geolocator = Nominatim(user_agent="railway_accident_geocoder")
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    max_retries=1,
    swallow_exceptions=True
)

geo_lookup = pd.DataFrame(geo_rows)

geo_lookup["geocode_success"] = geo_lookup["latitude"].notna()


In [14]:
geo_lookup.to_csv(
    "accident_location_geocoding.csv",
    index=False
)


In [15]:
acc = acc.merge(
    geo_lookup,
    on="location_clean",
    how="left"
)


In [ ]:
import geopandas as gpd

acc_gdf = gpd.GeoDataFrame(
    acc.dropna(subset=["longitude", "latitude"]),
    geometry=gpd.points_from_xy(acc.longitude, acc.latitude),
    crs="EPSG:4326"
).to_crs(epsg=27700)


In [16]:
print("Total accidents:", len(acc))
print("Likely UK:", acc["likely_uk"].sum())
print("Geocoded:", acc["latitude"].notna().sum())

acc.loc[acc["latitude"].isna(), "location"].value_counts().head(20)


Total accidents: 9212
Likely UK: 1902
Geocoded: 0


location
London Bridge            47
Preston                  47
Crewe                    41
Waterloo                 35
Carlisle                 33
Unknown location         32
Manchester Victoria      31
Victoria                 28
Kings Cross              28
Cannon Street            27
Wigan                    26
Birmingham New Street    25
Liverpool Street         25
York                     24
Leicester                23
Hatfield                 22
Accrington               22
Bolton                   21
Stafford                 21
Newcastle Central        21
Name: count, dtype: int64

## Uniform jurisdiction: 1851 Registration Districts

All locations (accidents, branches, stations) are assigned to **1851 Census Registration Districts** via spatial join. This gives:
- **Uniform geography** for census matching (1851, 1861 England & Wales censuses use RDs)
- **No reliance on geocoder resolution** — coordinates are snapped to the containing RD polygon

**Data source:** UK Data Service ReShare 852948 or Campop (1851EngWalesRegistrationDistrict.shp)
**Note:** 1851 RD boundaries cover England & Wales only. Scotland stations will have `rd_name` = NaN.

In [ ]:
# Load 1851 Registration District boundaries (England & Wales)
# Requires: download shapefile from UK Data Service https://reshare.ukdataservice.ac.uk/852948/
from pathlib import Path

rd_path = Path(rd_1851_shp)
if not rd_path.exists():
    raise FileNotFoundError(
        f"1851 RD shapefile not found at {rd_1851_shp}\n"
        "Download from UK Data Service: https://reshare.ukdataservice.ac.uk/852948/\n"
        "Or Campop (contact Dr Max Satchell): 1851EngWalesRegistrationDistrict.shp"
    )

rds = gpd.read_file(rd_1851_shp)
rds = rds.to_crs(epsg=27700)

# Inspect column names (vary by source)
print("RD shapefile columns:", list(rds.columns))
# Common: NAME, RD1851NM, or similar - we'll use the first non-geometry text column
rd_name_col = [c for c in rds.columns if c != "geometry" and rds[c].dtype == object][0]
print(f"Using '{rd_name_col}' as registration district identifier")

In [ ]:
def assign_to_rd(points_gdf, rds_gdf, rd_name_col="NAME"):
    """Spatial join: assign each point to its containing 1851 Registration District."""
    pts = points_gdf.to_crs(rds_gdf.crs)
    joined = gpd.sjoin(pts, rds_gdf[[rd_name_col, "geometry"]], how="left", predicate="within")
    # If within fails (e.g. boundary issues), try nearest neighbor
    missing = joined[rd_name_col].isna()
    if missing.any():
        nearest = gpd.sjoin_nearest(
            pts.loc[missing], rds_gdf[[rd_name_col, "geometry"]], how="left", max_distance=5000
        )
        nearest = nearest[~nearest.index.duplicated(keep="first")]
        joined.loc[nearest.index, rd_name_col] = nearest[rd_name_col].values
    return joined.drop(columns=["index_right"], errors="ignore")

In [ ]:
# --- ACCIDENTS: Load from geocoding cache ---
acc_loaded = pd.read_csv(accident_csv)
acc_loaded["_join"] = acc_loaded["location"].astype(str).str.lower().str.strip()

if not Path(accident_geo_cache).exists():
    raise FileNotFoundError(
        f"Accident geocoding cache not found: {accident_geo_cache}\n"
        "Run geocoding first (cells 5-7) or use the R pipeline to generate cache_accident_geocoding.csv"
    )
acc_geo = pd.read_csv(accident_geo_cache)
acc_geo["_join"] = acc_geo["location"].astype(str).str.lower().str.strip()
acc_geo = acc_geo.drop_duplicates(subset=["_join"])
acc_with_coords = acc_loaded.merge(acc_geo[["_join", "latitude", "longitude"]], on="_join", how="left")
acc_with_coords = acc_with_coords.dropna(subset=["latitude", "longitude"])
acc_gdf = gpd.GeoDataFrame(
    acc_with_coords,
    geometry=gpd.points_from_xy(acc_with_coords["longitude"], acc_with_coords["latitude"]),
    crs="EPSG:4326"
).to_crs(epsg=27700)

acc_with_rd = assign_to_rd(acc_gdf, rds, rd_name_col)
acc_with_rd = acc_with_rd.rename(columns={rd_name_col: "rd_name"})
print(f"Accidents with coordinates: {len(acc_with_rd)}")
print(f"Accidents assigned to 1851 RD: {acc_with_rd['rd_name'].notna().sum()} ({100*acc_with_rd['rd_name'].notna().mean():.1f}%)")

In [ ]:
# --- BRANCHES: Load from cache and assign to RD ---
br_raw = pd.read_csv(branch_csv).dropna(subset=["cleaned_loc"])
br_raw["_join"] = br_raw["cleaned_loc"].astype(str).str.lower().str.strip()

if not Path(union_geo_cache).exists():
    raise FileNotFoundError(f"Union geocoding cache not found: {union_geo_cache}")
br_geo = pd.read_csv(union_geo_cache)
br_geo["_join"] = br_geo["location"].astype(str).str.lower().str.strip()
br_geo = br_geo.drop_duplicates(subset=["_join"])
br_with_coords = br_raw.merge(br_geo[["_join", "latitude", "longitude"]], on="_join", how="left")
br_with_coords = br_with_coords.dropna(subset=["latitude", "longitude"])

br_gdf = gpd.GeoDataFrame(
    br_with_coords,
    geometry=gpd.points_from_xy(br_with_coords["longitude"], br_with_coords["latitude"]),
    crs="EPSG:4326"
).to_crs(epsg=27700)

br_with_rd = assign_to_rd(br_gdf, rds, rd_name_col)
br_with_rd = br_with_rd.rename(columns={rd_name_col: "rd_name"})
print(f"Branches with coordinates: {len(br_with_rd)}")
print(f"Branches assigned to 1851 RD: {br_with_rd['rd_name'].notna().sum()} ({100*br_with_rd['rd_name'].notna().mean():.1f}%)")

In [ ]:
# --- STATIONS: Assign to RD (stations already have geometry) ---
stns = gpd.read_file(station_shp).to_crs(epsg=27700)
stns_with_rd = assign_to_rd(stns, rds, rd_name_col)
stns_with_rd = stns_with_rd.rename(columns={rd_name_col: "rd_name"})
print(f"Stations: {len(stns_with_rd)}")
print(f"Stations assigned to 1851 RD (England & Wales): {stns_with_rd['rd_name'].notna().sum()}")
print(f"Stations in Scotland (no RD): {(stns_with_rd['rd_name'].isna()).sum()}")

In [ ]:
# --- RD-LEVEL AGGREGATES (for census merge) ---
# All three datasets now share rd_name. Aggregate counts per registration district.
rd_acc = acc_with_rd.dropna(subset=["rd_name"]).groupby("rd_name").size().reset_index(name="accident_count")
rd_br = br_with_rd.dropna(subset=["rd_name"]).groupby("rd_name").size().reset_index(name="branch_count")
rd_stn = stns_with_rd.dropna(subset=["rd_name"]).groupby("rd_name").size().reset_index(name="station_count")

rd_merged = rd_stn.merge(rd_acc, on="rd_name", how="left").merge(rd_br, on="rd_name", how="left")
rd_merged["accident_count"] = rd_merged["accident_count"].fillna(0).astype(int)
rd_merged["branch_count"] = rd_merged["branch_count"].fillna(0).astype(int)
print("Registration-district-level dataset (merge with 1851/1861 census on rd_name):")
print(rd_merged.head(10))

In [ ]:
# Save outputs (all with rd_name for census matching)
out_path = Path(base_path)
acc_with_rd.drop(columns=["geometry"], errors="ignore").to_csv(out_path / "accidents_with_rd1851.csv", index=False)
br_with_rd.drop(columns=["geometry"], errors="ignore").to_csv(out_path / "branches_with_rd1851.csv", index=False)
stns_with_rd.drop(columns=["geometry"], errors="ignore").to_csv(out_path / "stations_with_rd1851.csv", index=False)
rd_merged.to_csv(out_path / "rd1851_aggregates.csv", index=False)
print(f"Saved to {out_path}")

In [17]:
stations = gpd.read_file(station_shp)
print(stations.columns)


Index(['Id', 'geometry'], dtype='object')


In [19]:
STATION_NAME_COL = "Station"   # CHANGE to the correct column
stations = stations.to_crs(epsg=27700)


In [20]:
stations["station_clean"] = (
    stations[STATION_NAME_COL]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z ]", "", regex=True)
    .str.strip()
)


KeyError: 'Station'